In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# 1. Load Data
df = pd.read_csv('/content/prediction dataset.csv')
print("Original Data Shape:", df.shape)
print(df.head())

Original Data Shape: (100000, 9)
   gender   age  hypertension  heart_disease smoking_history    bmi  \
0  Female  80.0             0              1           never  25.19   
1  Female  54.0             0              0         No Info  27.32   
2    Male  28.0             0              0           never  27.32   
3  Female  36.0             0              0         current  23.45   
4    Male  76.0             1              1         current  20.14   

   HbA1c_level  blood_glucose_level  diabetes  
0          6.6                  140         0  
1          6.6                   80         0  
2          5.7                  158         0  
3          5.0                  155         0  
4          4.8                  155         0  


In [ ]:
# 2. Basic Info

print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())


Data Types:
 gender                  object
age                    float64
hypertension             int64
heart_disease            int64
smoking_history         object
bmi                    float64
HbA1c_level            float64
blood_glucose_level      int64
diabetes                 int64
dtype: object

Missing Values:
 gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64


In [ ]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns
target_col = 'diabetes'  # Define target_col early for consistent use

# Numerical: Replace with mean, only if there are missing values
for col in num_cols:
    if df[col].isnull().any():
        num_imputer = SimpleImputer(strategy='mean')
        df[col] = num_imputer.fit_transform(df[[col]])

# Categorical: Replace with mode, only if there are missing values
for col in cat_cols:
    if df[col].isnull().any():
        cat_imputer = SimpleImputer(strategy='most_frequent')
        df[col] = cat_imputer.fit_transform(df[[col]])

# Check missing values after handling
print(df.isnull().sum())

gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64


In [ ]:
# 4. Encode Categorical Variables
le = LabelEncoder()
current_cat_cols = df.select_dtypes(include=['object']).columns
binary_cols = [col for col in current_cat_cols if df[col].nunique() == 2]
for col in binary_cols:
    df[col] = le.fit_transform(df[col])
cols_for_ohe = df.select_dtypes(include=['object']).columns.tolist()
cols_for_ohe = [col for col in cols_for_ohe if df[col].nunique() > 2]

# One-hot Encoding (for more than 2 categories)
df = pd.get_dummies(df, columns=cols_for_ohe,drop_first=True)


In [ ]:
# 5. Outlier Detection and Treatment (using IQR)
for col in num_cols_for_outliers:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])


In [ ]:

scaler = StandardScaler()
df[selected_features.tolist()] = scaler.fit_transform(df[selected_features.tolist()])


In [ ]:
# 7. Feature Selection (if classification target is available)
target_col = 'diabetes'
if target_col in df.columns:
   # Features and target
    X = df.drop(target_col, axis=1)
    y = df[target_col]
    bestfeatures = SelectKBest(score_func=f_classif, k=10)
    fit = bestfeatures.fit(X, y)
    selected_features = X.columns[fit.get_support()]
    df = df[selected_features.to_list() + [target_col]]
    print("\nSelected Features:", selected_features.to_list())


Selected Features: ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'gender_Male', 'smoking_history_current', 'smoking_history_ever', 'smoking_history_former', 'smoking_history_never', 'smoking_history_not current']


In [ ]:
X = np.asarray(df[selected_features])
y = np.asarray(df[target_col])

X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size = 0.3, random_state = 4)

print ('Train set:', X_train.shape, y_train.shape)
print ('Test set:', X_test.shape, y_test.shape)

Train set: (70000, 10) (70000,)
Test set: (30000, 10) (30000,)


In [ ]:
df.columns


Index(['age', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'gender_Male',
       'smoking_history_current', 'smoking_history_ever',
       'smoking_history_former', 'smoking_history_never',
       'smoking_history_not current', 'diabetes'],
      dtype='object')

In [ ]:
# 1. Logistic Regression
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Logistic Regression Accuracy:", model.score(X_test, y_test))


Logistic Regression Accuracy: 0.9578666666666666


In [ ]:
# 2. K-Nearest Neighbors
from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("KNN Accuracy:", model.score(X_test, y_test))

KNN Accuracy: 0.9628333333333333


In [ ]:
# 3. Decision Tree
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Decision Tree Accuracy:", model.score(X_test, y_test))


Decision Tree Accuracy: 0.9505333333333333


In [ ]:
# 4. Random Forest
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Random Forest Accuracy:", model.score(X_test, y_test))


Random Forest Accuracy: 0.9702


In [ ]:
# 5. Support Vector Machine
from sklearn.svm import SVC
model = SVC()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("SVM Accuracy:", model.score(X_test, y_test))

SVM Accuracy: 0.9637333333333333


In [ ]:
# 6. Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier
model = GradientBoostingClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Gradient Boosting Accuracy:", model.score(X_test, y_test))

Gradient Boosting Accuracy: 0.9727


In [ ]:
# 7. Naive Bayes
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Naive Bayes Accuracy:", model.score(X_test, y_test))

Naive Bayes Accuracy: 0.9331333333333334
